[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Bulk Writes and Transactions


## What you will be able to do

Send many writes in one round trip with `bulk_write`, choose between `ordered` and `unordered`
knowing what each leaves behind when one operation fails, and find the real error, which is not in
the exception's message. Empty a collection two ways and say what happens to its indexes. Run a
transaction, and recognize the write inside one that was never part of it. And say why a plain
`mongod` refuses transactions at all.


## The idea

### The problem

Writes are individually atomic and collectively nothing. Ten `update_one` calls are ten operations,
ten round trips, and ten chances to stop halfway, and MongoDB will not put them back if the eighth
one fails.

`bulk_write` fixes the round trips and not the halfway. Transactions fix the halfway and cost
something real.

### What a bulk write is

A list of operations, sent together, applied by the server in one pass. It is not a transaction: if
one fails, the others still happened. What `ordered` decides is only whether the server stops at the
first failure or carries on.

### What a transaction is

A session, and every write that names it. Either all of them land or none of them do. The words
"that names it" are the whole difficulty: a write inside the block that does not take `session=` is
an ordinary write, and it survives the abort.

### Why a plain mongod refuses

Transactions are built on the replica set oplog, so a server with no replica set has nowhere to
record them. A single node replica set is enough, which is what this guide's boot cell makes, and it
is why `rs.initiate` is in there.

### Where this shows up

Any import, any batch job, any operation that touches two documents and has to be all or nothing.
Also the first time somebody copies a transaction example onto a development machine running a plain
`mongod`.

### What this notebook covers

`bulk_write`, ordered and not, and `BulkWriteError.details`. `delete_many({})` against `drop()`.
Transactions with `with_transaction`, the session that has to be passed everywhere, and the
standalone that says no. Then the four failures.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

shop.accounts.drop()
shop.accounts.insert_many([{"_id": "a", "n": 100}, {"_id": "b", "n": 0}])


def transfer(session):
    shop.accounts.update_one({"_id": "a"}, {"$inc": {"n": -10}}, session=session)
    shop.accounts.update_one({"_id": "b"}, {"$inc": {"n": 10}})     # session= forgotten
    raise RuntimeError("and then something failed")


with client.start_session() as session:
    try:
        session.with_transaction(transfer)
    except RuntimeError as error:
        print("the transaction aborted:", error)

print("after the abort:", list(shop.accounts.find().sort("_id")))
print("a lost nothing, and b gained ten out of thin air")
client.close()
```

```
the transaction aborted: and then something failed
after the abort: [{'_id': 'a', 'n': 100}, {'_id': 'b', 'n': 10}]
a lost nothing, and b gained ten out of thin air
```

The transaction did its job: the write that named the session was rolled back. The other one was
never in the transaction at all, so there was nothing to roll back, and ten units of money now exist
that did not before.


## Setup

Eight imports, two MongoDB servers, the boot cell, and four helpers.

- `pymongo` with `InsertOne`, `UpdateOne`, `ReplaceOne` and `DeleteOne`, which are what a bulk
  write is made of
- `subprocess` and `os` install and start the servers, `sys` names this Python, `time` waits
- `random` seeds the data the same way every run, with `version` and `PackageNotFoundError`

This Setup starts a **second** server, on port 27018, with no replica set, because the only honest
way to show what a plain `mongod` refuses is to have one. `accounts` puts two accounts back the way
they started and `balances` reads them. `failed` prints a failure's message without the parts that
change between runs.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo
from pymongo import DeleteOne, InsertOne, ReplaceOne, UpdateOne

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

STANDALONE = "mongodb://127.0.0.1:27018/shop"                       # started below, no replica set


def failed(error):
    """A failure's real message, without the parts that change every run."""
    details = getattr(error, "details", None) or {}
    return f"{type(error).__name__}: {details.get('errmsg', str(error).split(', full error')[0])}"


def start_standalone(wait=30):
    """A second mongod with no --replSet, so this notebook can show what it refuses to do."""
    try:
        with pymongo.MongoClient(STANDALONE + "?directConnection=true",
                                 serverSelectionTimeoutMS=1500) as probe:
            probe.admin.command("ping")
            return "already running"
    except pymongo.errors.PyMongoError:
        pass
    if sys.platform != "linux" and shell("which mongod")[0] != 0:
        return "not available here, and the section below says what it would have shown"

    path = f"{DBPATH}-standalone"
    os.makedirs(path, exist_ok=True)
    code, out = shell(f"mongod --dbpath {path} --port 27018 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {path}.log")
    if code != 0:
        print("  the standalone did not start:", shell(f"tail -5 {path}.log")[1][-200:])
        return "unavailable"
    for _ in range(wait):
        try:
            with pymongo.MongoClient(STANDALONE + "?directConnection=true",
                                     serverSelectionTimeoutMS=1500) as probe:
                probe.admin.command("ping")
                return "started on port 27018"
        except pymongo.errors.PyMongoError:
            time.sleep(1)
    return "unavailable"


def accounts():
    """Two accounts, a hundred in one and nothing in the other, put back every time."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        shop.accounts.drop()
        shop.accounts.insert_many([{"_id": "a", "n": 100}, {"_id": "b", "n": 0}])
        return {row["_id"]: row["n"] for row in shop.accounts.find().sort("_id")}


def balances():
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        return {row["_id"]: row["n"] for row in shop.accounts.find().sort("_id")}


print("server:    ", start_server())
print("replica:   ", initiate())
print("standalone:", start_standalone())
print("seeded:    ", seed(), "products")
print("accounts:  ", accounts())
print(report())


server:     already running
replica:    replica set rs0, primary
standalone: already running
seeded:     500 products
accounts:   {'a': 100, 'b': 0}
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### Many writes, one round trip


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.bulk.drop()

result = shop.bulk.bulk_write([
    InsertOne({"_id": 1, "n": 1}),
    InsertOne({"_id": 2, "n": 2}),
    UpdateOne({"_id": 1}, {"$inc": {"n": 10}}),
    ReplaceOne({"_id": 2}, {"n": 99}),
    DeleteOne({"_id": 3}),                                          # matches nothing, and that is fine
])

print("inserted:", result.inserted_count, "| modified:", result.modified_count,
      "| deleted:", result.deleted_count)
print("what is there:", list(shop.bulk.find().sort("_id")))


inserted: 2 | modified: 2 | deleted: 0
what is there: [{'_id': 1, 'n': 11}, {'_id': 2, 'n': 99}]


Five operations, one request. The counts come back separately per kind, and an operation matching
nothing is not an error: `DeleteOne` that matched nothing simply deleted nothing.

### ordered, and what is left behind

The default stops at the first failure:


In [3]:
operations = [InsertOne({"_id": 1}), InsertOne({"_id": 2}),
              InsertOne({"_id": 2}),                                # the duplicate
              InsertOne({"_id": 3})]

shop.bulk.drop()
try:
    shop.bulk.bulk_write(operations)                                # ordered=True by default
except pymongo.errors.BulkWriteError:
    print("ordered:   wrote", shop.bulk.count_documents({}), "of 4, and never tried the fourth")

shop.bulk.drop()
try:
    shop.bulk.bulk_write(operations, ordered=False)
except pymongo.errors.BulkWriteError:
    print("unordered: wrote", shop.bulk.count_documents({}), "of 4, and tried all of them")


ordered:   wrote 2 of 4, and never tried the fourth
unordered: wrote 3 of 4, and tried all of them


Two and three. `ordered=True` is the default and is what you want when a later operation depends on
an earlier one. `ordered=False` is what you want for an import, where every row is independent and
you would rather load what you can and be told what you could not.

Neither is a transaction. Both left documents behind.

### Where the real error is


In [4]:
shop.bulk.drop()
try:
    shop.bulk.bulk_write(operations, ordered=False)
except pymongo.errors.BulkWriteError as error:
    print("what printing the exception gives you:")
    print("  ", str(error).split(", full error")[0])
    print()
    print("what you actually need:")
    for problem in error.details["writeErrors"]:
        print("   index", problem["index"], "| code", problem["code"], "|",
              problem["errmsg"].split(", full error")[0][:70])


what printing the exception gives you:
   batch op errors occurred

what you actually need:
   index 2 | code 11000 | E11000 duplicate key error collection: shop.bulk index: _id_ dup key: 


`batch op errors occurred` is all the message says, and a program that logs `str(error)` has thrown
away everything useful. `details["writeErrors"]` has one entry per failed operation, each naming its
**index in the list you sent**, which is how you find out which of ten thousand rows was bad.

`details` also carries `nInserted` and the other counts, so a partial success can be reported
accurately rather than as a failure.

### Emptying a collection

Two ways, and they differ in more than speed:


In [5]:
shop.big.drop()
shop.big.insert_many([{"n": number} for number in range(20000)])
shop.big.create_index("n")
print("indexes:", [index["name"] for index in shop.big.list_indexes()])

shop.big.delete_many({})
print("after delete_many({}): ", shop.big.count_documents({}), "documents, indexes",
      [index["name"] for index in shop.big.list_indexes()])


indexes: ['_id_', 'n_1']
after delete_many({}):  0 documents, indexes ['_id_', 'n_1']


`delete_many({})` removes the documents one at a time, writes every deletion to the oplog, and
leaves the collection and its indexes standing. That is what you want when something is replicating
or watching, and it is the slower of the two by a wide margin on a big collection.


In [6]:
shop.big.insert_many([{"n": number} for number in range(20000)])
shop.big.create_index("n")

shop.big.drop()
print("after drop():", shop.big.count_documents({}), "documents, indexes",
      [index["name"] for index in shop.big.list_indexes()])
print("the collection and every index it had are gone, and must be created again")


after drop(): 0 documents, indexes []
the collection and every index it had are gone, and must be created again


`drop()` removes the collection itself. It is close to instant however large the collection is, and
it takes the indexes with it, which is the part people forget: the next write recreates the
collection with only its `_id` index, and every query that relied on another one is now a
collection scan.

### A transaction, with the session everywhere it belongs


In [7]:
print("before:", accounts())


def transfer(session, amount=10):
    """Every write names the session, which is what puts it in the transaction."""
    shop.accounts.update_one({"_id": "a"}, {"$inc": {"n": -amount}}, session=session)
    shop.accounts.update_one({"_id": "b"}, {"$inc": {"n": amount}}, session=session)


with client.start_session() as session:
    session.with_transaction(transfer)

print("after: ", balances(), "<- both, or neither")


before: {'a': 100, 'b': 0}
after:  {'a': 90, 'b': 10} <- both, or neither


`with_transaction` starts the transaction, runs the callback, commits, and retries the whole thing
if the server reports a transient failure. That retry is why the callback must be safe to run twice,
and it is the reason to use `with_transaction` rather than `start_transaction` by hand.

Now the same thing failing:


In [8]:
accounts()


def transfer_then_fail(session):
    shop.accounts.update_one({"_id": "a"}, {"$inc": {"n": -10}}, session=session)
    shop.accounts.update_one({"_id": "b"}, {"$inc": {"n": 10}}, session=session)
    raise RuntimeError("the receipt could not be written")


with client.start_session() as session:
    try:
        session.with_transaction(transfer_then_fail)
    except RuntimeError as error:
        print("aborted:", error)

print("after: ", balances(), "<- unchanged, which is the whole point")


aborted: the receipt could not be written
after:  {'a': 100, 'b': 0} <- unchanged, which is the whole point


### What a plain mongod says

The second server started in Setup has no replica set:


In [9]:
standalone = pymongo.MongoClient(STANDALONE, tz_aware=True)
standalone.get_default_database().accounts.replace_one({"_id": "a"}, {"_id": "a", "n": 1},
                                                       upsert=True)
print("it works as a database:",
      standalone.get_default_database().accounts.count_documents({}))

try:
    with standalone.start_session() as session:
        with session.start_transaction():
            standalone.get_default_database().accounts.update_one(
                {"_id": "a"}, {"$inc": {"n": 1}}, session=session)
except pymongo.errors.OperationFailure as error:
    print("but:", failed(error))


it works as a database: 1
but: OperationFailure: Transaction numbers are only allowed on a replica set member or mongos


Transactions need the oplog, and the oplog is a replica set feature. A single node replica set is
enough and costs nothing extra, which is why this guide's boot cell runs `replSetInitiate` rather
than leaving you with a plain `mongod`.

### When to reach for which

| What you want | How to write it |
|---|---|
| many writes, one round trip | `bulk_write([...])` |
| stop at the first failure | `ordered=True`, the default |
| do everything possible | `ordered=False` |
| which operation failed | `error.details["writeErrors"]`, by `index` |
| all or nothing | `session.with_transaction(callback)` |
| a write inside a transaction | pass `session=session`, every time |
| to empty a collection, keeping indexes | `delete_many({})` |
| to remove a collection entirely | `drop()`, and recreate the indexes |
| one document changed atomically | nothing special: it already is |

The default is no transaction. One document's update is already atomic, so a transaction is for the
case where two documents have to agree, and it costs a session, a retry loop, and a replica set.

### A transfer that cannot go wrong quietly, finished


In [10]:
def move(client, source, target, amount):
    """All of it or none of it, with the session on every write and the balance checked inside."""
    shop = client.get_default_database()

    def body(session):
        taken = shop.accounts.find_one_and_update(
            {"_id": source, "n": {"$gte": amount}},                 # only if there is enough
            {"$inc": {"n": -amount}},
            session=session)
        if taken is None:
            raise ValueError(f"{source} has less than {amount}")
        shop.accounts.update_one({"_id": target}, {"$inc": {"n": amount}}, session=session)

    with client.start_session() as session:
        session.with_transaction(body)
    return balances()


accounts()
print("move 30:", move(client, "a", "b", 30))

try:
    move(client, "a", "b", 1000)
except ValueError as error:
    print("move 1000:", error)
print("after the refusal:", balances(), "<- nothing moved")


move 30: {'a': 70, 'b': 30}
move 1000: a has less than 1000
after the refusal: {'a': 70, 'b': 30} <- nothing moved


Two guards, and they do different jobs. The `{"n": {"$gte": amount}}` in the filter means the
withdrawal cannot take an account below zero even under concurrency. The transaction means the
deposit cannot happen without the withdrawal.

Raising inside the callback aborts the transaction, so the `ValueError` both reports the problem and
undoes the half that had already been done.

### Where each part came from

| In `move` | What it relies on | The section that showed it |
|---|---|---|
| `with_transaction(body)` | all or nothing, with retries | A transaction, with the session |
| `session=session` on both writes | a write without it is not in the transaction | A first look |
| `find_one_and_update` with a condition | read and change as one operation | **Update Operators** |
| raising to abort | an exception leaving the block unaborted | A transaction, with the session |
| a replica set at all | transactions needing the oplog | What a plain mongod says |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/07-bulk-writes-and-transactions-solutions.ipynb).

**1.** Send an insert, an update and a delete in one `bulk_write`.


In [11]:
# your code here


**2.** Run the same failing batch ordered and unordered, and count what each wrote.


In [12]:
# your code here


**3.** Print which operation in a batch failed, by its index.


In [13]:
# your code here


**4.** Show that `drop()` removes an index and `delete_many({})` does not.


In [14]:
# your code here


**5.** Move ten from one account to the other inside a transaction.


In [15]:
# your code here


**6.** Try a transaction against the standalone on port 27018.


In [16]:
# your code here


## Common errors

### pymongo.errors.BulkWriteError: batch op errors occurred


In [17]:
shop.bulk.drop()
shop.bulk.bulk_write([InsertOne({"_id": 1}), InsertOne({"_id": 1})])


BulkWriteError: batch op errors occurred, full error: {'writeErrors': [{'index': 1, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: shop.bulk index: _id_ dup key: { _id: 1 }', 'keyPattern': {'_id': 1}, 'keyValue': {'_id': 1}, 'op': {'_id': 1}}], 'writeConcernErrors': [], 'nInserted': 1, 'nUpserted': 0, 'nMatched': 0, 'nModified': 0, 'nRemoved': 0, 'upserted': []}

The traceback carries the details because PyMongo puts them in the string, which is generous but not
something to rely on: in a log the line is truncated, and in code `str(error)` gives you only the
first clause.

Read `details`, and report it:


In [18]:
shop.bulk.drop()
try:
    shop.bulk.bulk_write([InsertOne({"_id": 1}), InsertOne({"_id": 1}),
                          InsertOne({"_id": 2})], ordered=False)
except pymongo.errors.BulkWriteError as error:
    print("inserted:", error.details["nInserted"])
    for problem in error.details["writeErrors"]:
        print("  operation", problem["index"], "failed with code", problem["code"])


inserted: 2
  operation 1 failed with code 11000


### pymongo.errors.OperationFailure: Transaction numbers are only allowed on a replica set member


In [19]:
try:
    with standalone.start_session() as session:
        with session.start_transaction():
            standalone.get_default_database().accounts.insert_one({"x": 1}, session=session)
except pymongo.errors.OperationFailure as error:
    print(failed(error))


OperationFailure: Transaction numbers are only allowed on a replica set member or mongos


The usual way to meet this is a development machine running `mongod` the way the installation
instructions say, against code written for a cluster. Nothing in the code is wrong.

The fix is one flag and one command, which is exactly what this guide's boot cell does:
`--replSet rs0` when starting the server, and `replSetInitiate` once. A single node is a legitimate
replica set.

### No error: the write that was never in the transaction


In [20]:
accounts()


def half_in(session):
    shop.accounts.update_one({"_id": "a"}, {"$inc": {"n": -10}}, session=session)
    shop.accounts.update_one({"_id": "b"}, {"$inc": {"n": 10}})     # no session
    raise RuntimeError("and then it failed")


with client.start_session() as session:
    try:
        session.with_transaction(half_in)
    except RuntimeError:
        pass

print("after the abort:", balances())
print("the rollback worked perfectly on the write that was in the transaction")


after the abort: {'a': 100, 'b': 10}
the rollback worked perfectly on the write that was in the transaction


This is the failure the notebook exists for. Everything about it looks right: there is a session,
there is a transaction, the abort happened, and the rollback did exactly what it promised for the
write that had opted in.

There is no way to make MongoDB catch this, because a write without a session is a perfectly normal
write. The defenses are to give the callback the session as its only argument, as
`with_transaction` does, and to never reach for a collection object from an enclosing scope inside
one:


In [21]:
def careful(session):
    """The session is the only way in, so forgetting it is a NameError rather than a silent write."""
    collection = session.client.get_default_database().accounts
    collection.update_one({"_id": "a"}, {"$inc": {"n": -10}}, session=session)
    collection.update_one({"_id": "b"}, {"$inc": {"n": 10}}, session=session)


accounts()
with client.start_session() as session:
    session.with_transaction(careful)
print("both moved:", balances())


both moved: {'a': 90, 'b': 10}


### No error: the bulk write that half happened


In [22]:
shop.bulk.drop()
try:
    shop.bulk.bulk_write([InsertOne({"_id": number}) for number in (1, 2, 2, 3, 4)])
except pymongo.errors.BulkWriteError:
    pass

print("documents left behind:", sorted(row["_id"] for row in shop.bulk.find()))
print("the exception was caught and ignored, and the collection is now half loaded")


documents left behind: [1, 2]
the exception was caught and ignored, and the collection is now half loaded


`bulk_write` is not a transaction and never claimed to be. An import that catches
`BulkWriteError` and logs it has a collection in a state nobody chose: the first two rows are in and
the rest are not.

Either make the operations idempotent so rerunning finishes the job, which is what
`ReplaceOne(..., upsert=True)` gives you, or wrap the batch in a transaction and accept the cost:


In [23]:
shop.bulk.drop()
rows = [{"_id": number, "n": number} for number in (1, 2, 2, 3, 4)]

shop.bulk.bulk_write([ReplaceOne({"_id": row["_id"]}, row, upsert=True) for row in rows])
print("first run: ", sorted(row["_id"] for row in shop.bulk.find()))

shop.bulk.bulk_write([ReplaceOne({"_id": row["_id"]}, row, upsert=True) for row in rows])
print("second run:", sorted(row["_id"] for row in shop.bulk.find()), "<- and no error either time")


first run:  [1, 2, 3, 4]
second run: [1, 2, 3, 4] <- and no error either time


In [24]:
shop.bulk.drop()
shop.big.drop()
client.close()
standalone.close()
print("tidied up and closed")


tidied up and closed


## Recap

- `bulk_write` sends many operations in one round trip. It is **not** a transaction: a failure
  leaves the others applied.
- `ordered=True`, the default, stops at the first failure. `ordered=False` attempts everything and
  reports all the failures at the end.
- `BulkWriteError`'s message says only `batch op errors occurred`. The useful part is
  `details["writeErrors"]`, and each entry names the failing operation's index in your list.
- `delete_many({})` removes documents and keeps the collection and its indexes. `drop()` removes
  everything including the indexes, is far faster, and leaves you to create them again.
- A transaction is a session plus `with_transaction`, and every write in it must be given
  `session=`. A write without one is an ordinary write and survives the abort, silently.
- `with_transaction` retries on transient failures, so its callback must be safe to run twice.
- Transactions need a replica set. A plain `mongod` answers
  `Transaction numbers are only allowed on a replica set member or mongos`, and a single node
  replica set is enough.


## What is next

**Indexes** is why some of these queries are fast: compound key order, `explain` and the
`COLLSCAN` it reveals, the unique index that will not build over data that already has duplicates,
and the TTL index whose deletions arrive about a minute late.


---

&#8592; **Previous:** [Update Operators](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/06-update-operators.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Indexes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/08-indexes.ipynb) &#8594;
